<a href="https://colab.research.google.com/github/dptel22/Nosh-Robotics/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Pretrained Segmentation Model (SAM 3) — Setup & Audit
**Phase 1: Setup Only**
- GPU Baseline (`nvidia-smi`, CUDA capability, total VRAM)
- Runtime Environment Versions (Python, PyTorch, TorchVision, Transformers)
- Repository Clone & Package Installation (`facebookresearch/sam3`)
- Hugging Face Authentication & Gated Access Verification (`facebook/sam3`)
- Official Checkpoint Download (`sam3.pt` / `model.safetensors`)
- Recorded Audit & Exact Package Versions

> **Rule Enforcement:** Strictly NO images are processed during Phase 1.

In [ ]:
# Cell 1: Environment & GPU Baseline Check
import os
import sys
import subprocess

print('=' * 60)
print('1. NVIDIA GPU BASELINE')
print('=' * 60)
try:
    smi_output = subprocess.check_output(['nvidia-smi'], text=True)
    print(smi_output)
except Exception as e:
    print(f'nvidia-smi check failed: {e}')

print('=' * 60)
print('2. RUNTIME ENVIRONMENT VERSIONS')
print('=' * 60)
print(f'Python Version: {sys.version}')

import torch
import torchvision
try:
    import transformers
    transformers_ver = transformers.__version__
except ImportError:
    transformers_ver = 'Not installed yet'

print(f'PyTorch Version: {torch.__version__}')
print(f'TorchVision Version: {torchvision.__version__}')
print(f'Transformers Version: {transformers_ver}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA Version (torch): {torch.version.cuda}')
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'Device Capability: {torch.cuda.get_device_capability(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM: {total_mem:.2f} GB')


In [ ]:
# Cell 2: Clone Official SAM 3 Repository & Install Dependencies
import os
import sys
import subprocess

print('=' * 60)
print('3. CLONING OFFICIAL SAM 3 REPOSITORY')
print('=' * 60)

SAM3_DIR = '/content/sam3'
if not os.path.exists(SAM3_DIR):
    res = subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam3.git', SAM3_DIR], capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)
else:
    print(f'Repository already exists at {SAM3_DIR}')

# Record git commit hash
commit_hash = subprocess.check_output(['git', '-C', SAM3_DIR, 'rev-parse', 'HEAD'], text=True).strip()
print(f'Official SAM 3 Repository Commit: {commit_hash}')

print('\n' + '=' * 60)
print('4. INSTALLING SAM 3 & PREREQUISITES')
print('=' * 60)

# Install huggingface_hub
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'huggingface_hub', 'safetensors'])

# Install SAM 3 package in editable mode
install_res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', SAM3_DIR], capture_output=True, text=True)
print(install_res.stdout)
if install_res.stderr:
    print(install_res.stderr)
if install_res.returncode != 0:
    print('Warning: pip install -e . returned non-zero. Review output above.')
else:
    print('SAM 3 installation completed successfully.')


In [ ]:
# Cell 3: Hugging Face Authentication & Gated Checkpoint Access Check
import os
from huggingface_hub import HfApi, login

print('=' * 60)
print('5. HUGGING FACE AUTHENTICATION & ACCESS VERIFICATION')
print('=' * 60)

# Check for Colab userdata secret or environment variable HF_TOKEN
token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        print('Retrieved HF_TOKEN from Colab Secrets.')
    except Exception:
        pass

if token:
    login(token=token)
    print('Successfully authenticated with Hugging Face via token.')
else:
    print('[ATTENTION REQUIRED]: HF_TOKEN is not set.')
    print('Set HF_TOKEN in Colab Secrets (key icon on left sidebar) or run huggingface_hub.login().')

api = HfApi()
repo_id = 'facebook/sam3'
print(f'\nChecking gated repository access for: {repo_id}...')
try:
    repo_info = api.repo_info(repo_id=repo_id)
    print(f'SUCCESS: Access confirmed for {repo_id}!')
    print(f'Model ID: {repo_info.id}')
    print(f'Last Commit SHA: {repo_info.sha}')
    files = [f.rfilename for f in repo_info.siblings]
    print(f'Available files in repo ({len(files)} total):')
    for f in files:
        if f.endswith(('.pt', '.pth', '.safetensors', '.json', '.yaml', '.md')):
            print(f'  - {f}')
except Exception as e:
    print(f'ERROR: Cannot access {repo_id}: {e}')
    print('\nTroubleshooting:')
    print('1. Visit https://huggingface.co/facebook/sam3 while logged in.')
    print('2. Accept the Meta license terms and request access.')
    print('3. Ensure your HF token has read permissions and your account is approved.')


In [ ]:
# Cell 4: Download Official Checkpoint Only (NO IMAGES PROCESSED)
import os
from huggingface_hub import hf_hub_download

print('=' * 60)
print('6. OFFICIAL CHECKPOINT DOWNLOAD')
print('=' * 60)

repo_id = 'facebook/sam3'
ckpt_filename = 'sam3.pt'
print(f"Attempting download of official checkpoint '{ckpt_filename}' from '{repo_id}'...")

try:
    ckpt_path = hf_hub_download(
        repo_id=repo_id,
        filename=ckpt_filename,
        local_dir='/content/sam3_checkpoints'
    )
    size_mb = os.path.getsize(ckpt_path) / (1024 ** 2)
    print('SUCCESS: Checkpoint downloaded!')
    print(f'Local Path: {ckpt_path}')
    print(f'File Size: {size_mb:.2f} MB')
except Exception as e:
    print(f'Download of {ckpt_filename} failed: {e}')
    print("Attempting fallback to 'model.safetensors'...")
    try:
        ckpt_path = hf_hub_download(
            repo_id=repo_id,
            filename='model.safetensors',
            local_dir='/content/sam3_checkpoints'
        )
        size_mb = os.path.getsize(ckpt_path) / (1024 ** 2)
        print("SUCCESS: 'model.safetensors' downloaded!")
        print(f'Local Path: {ckpt_path}')
        print(f'File Size: {size_mb:.2f} MB')
    except Exception as e2:
        print(f'ERROR: Checkpoint download failed: {e2}')


In [ ]:
# Cell 5: Package Audit & Environment Snapshot
import sys
import subprocess
import json

print('=' * 60)
print('7. PACKAGE VERSION AUDIT')
print('=' * 60)

cmd = [sys.executable, '-m', 'pip', 'list', '--format=json']
packages = json.loads(subprocess.check_output(cmd, text=True))
key_pkgs = ['torch', 'torchvision', 'transformers', 'sam3', 'huggingface-hub', 'safetensors', 'accelerate', 'timm', 'numpy', 'opencv-python', 'pillow']

print(f"{'Package':<25} | {'Version'}")
print('-' * 40)
pkg_dict = {p['name'].lower(): p['version'] for p in packages}
for k in key_pkgs:
    ver = pkg_dict.get(k.lower(), 'NOT INSTALLED')
    print(f'{k:<25} | {ver}')

print('=' * 60)
print('PHASE 1 SETUP COMPLETE. NO IMAGES RUN.')
print('=' * 60)
